<a href="https://colab.research.google.com/github/GitGirlie27/urdu-ocr-codesaviours-si26-Amna/blob/main/SI26_Week5_Amna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INSTALL REQUIRED PACKAGES

In [1]:
!pip uninstall -y transformers tokenizers sentencepiece

!pip install -q \
transformers==4.46.3 \
tokenizers==0.20.3 \
sentencepiece==0.2.0 \
accelerate==1.0.1 \
datasets==3.1.0 \
evaluate==0.4.3 \
jiwer==3.0.5

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: sentencepiece 0.2.2
Uninstalling sentencepiece-0.2.2:
  Successfully uninstalled sentencepiece-0.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.4

# Check Versions

In [2]:
import transformers
import tokenizers
import sentencepiece

print(transformers.__version__)
print(tokenizers.__version__)
print(sentencepiece.__version__)

4.46.3
0.20.3
0.2.0


# 2. IMPORT LIBRARIES

In [3]:
import os
import gc
import cv2
import math
import random
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split

from transformers import (
    AutoImageProcessor,
    AutoTokenizer,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
    EarlyStoppingCallback
)

import evaluate

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch :", torch.__version__)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


Torch : 2.11.0+cu128
Device : cuda
GPU : Tesla T4


# Extract Dataset

In [4]:
ZIP_PATH = "/content/Urdu OCR Dataset Updated.zip"

EXTRACT_PATH = "/content"

if os.path.exists(ZIP_PATH):

    print("Extracting dataset...")

    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)

    print("Dataset extracted.")

else:

    print("Dataset already extracted OR ZIP file not found.")

print("\nFolders inside /content :\n")

print(os.listdir("/content"))

Extracting dataset...
Dataset extracted.

Folders inside /content :

['.config', 'Labels.csv', 'Urdu OCR Dataset Updated.zip', 'Urdu OCR Dataset', 'sample_data']


# 5. Processed Dataset

In [5]:
RAW_IMAGES_DIR = "/content/Urdu OCR Dataset"
PROCESSED_DIR = "/content/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

SUPPORTED = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


def preprocess_image(image):

    image = np.array(image)

    # Convert grayscale → RGB

    if len(image.shape) == 2:

        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    # Denoising

    image = cv2.fastNlMeansDenoisingColored(
        image,
        None,
        3,
        3,
        7,
        21
    )

    # CLAHE

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    l = clahe.apply(l)

    lab = cv2.merge((l,a,b))

    image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return Image.fromarray(image)


count = 0

for root, _, files in os.walk(RAW_IMAGES_DIR):

    for file in files:

        if file.lower().endswith(SUPPORTED):

            src = os.path.join(root, file)

            dst = os.path.join(PROCESSED_DIR, file)

            img = Image.open(src).convert("RGB")

            img = preprocess_image(img)

            img.save(dst)

            count += 1

print("Processed Images :", count)
print("Saved to :", PROCESSED_DIR)

Processed Images : 209
Saved to : /content/processed


# Load CSV File

In [6]:
CSV_PATH = "/content/Labels.csv"

df = pd.read_csv(
    CSV_PATH,
    encoding="utf-8-sig"
)

print(df.head())

print("\nDataset Shape :", df.shape)

# ----------------------------------------
# Basic Cleaning
# ----------------------------------------

df = df.dropna()

df = df.drop_duplicates()

df["Text"] = df["Text"].astype(str)

df = df[df["Text"].str.strip() != ""]

# ----------------------------------------
# Verify Image Paths
# ----------------------------------------

missing = 0

valid_rows = []

for _, row in df.iterrows():

    if os.path.exists(row["Image_no"]):

        valid_rows.append(row)

    else:

        missing += 1

df = pd.DataFrame(valid_rows)

df.reset_index(drop=True, inplace=True)

print("\nMissing Images :", missing)

print("Final Dataset :", df.shape)

# ----------------------------------------
# Train / Validation / Test Split
# ----------------------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Training   :", len(train_df))
print("Validation :", len(valid_df))
print("Testing    :", len(test_df))

train_df.reset_index(drop=True, inplace=True)
valid_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

                   Image_no                                               Text
0  /content/processed/0.png  پشاور، بنوں (نما ئندہ جنگ، اے ایف پی) بنوں میں...
1  /content/processed/1.png        اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس
2  /content/processed/2.png  کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ پولیس تشدد سے ا...
3  /content/processed/3.png  منشور علی جاں بحق اور 2 خواتین سمیت 14 افراد ز...
4  /content/processed/4.png  علاقے میں کرفیو نافذ کر دیا گیا ہے۔ مکینوں نے ...

Dataset Shape : (208, 2)

Missing Images : 0
Final Dataset : (208, 2)
Training   : 145
Validation : 31
Testing    : 32


# 6. Load Processor & Model

In [7]:
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

MODEL_NAME = "microsoft/trocr-base-printed"

print("Loading Processor...")

processor = TrOCRProcessor.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

print("Loading Model...")

model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

# Generation Configuration
model.config.max_length = 128
model.config.num_beams = 4
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0

print("Processor & Model Loaded Successfully")

Loading Processor...


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processor & Model Loaded Successfully


# Dataset Class

In [8]:
class UrduOCRDataset(Dataset):

    def __init__(
        self,
        dataframe,
        processor,
        max_target_length=128
    ):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row["Image_no"]).convert("RGB")

        # Image Processing
        pixel_values = self.processor(
            images=image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        # Text Tokenization
        labels = self.processor.tokenizer(
            row["Text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_target_length,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# Create Dataset Objects

In [9]:
train_dataset = UrduOCRDataset(
    dataframe=train_df,
    processor=processor,
    max_target_length=128
)

valid_dataset = UrduOCRDataset(
    dataframe=valid_df,
    processor=processor,
    max_target_length=128
)

test_dataset = UrduOCRDataset(
    dataframe=test_df,
    processor=processor,
    max_target_length=128
)

print(f"Train Samples      : {len(train_dataset)}")
print(f"Validation Samples : {len(valid_dataset)}")
print(f"Test Samples       : {len(test_dataset)}")

Train Samples      : 145
Validation Samples : 31
Test Samples       : 32


# Data Collator

In [10]:
from transformers import default_data_collator

data_collator = default_data_collator

# 8. Evaluation Metrics

In [11]:
import numpy as np
import evaluate
from jiwer import cer, wer

cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")


def normalized_edit_distance(predictions, references):

    scores = []

    for pred, ref in zip(predictions, references):

        if len(ref) == 0:
            continue

        distance = cer(ref, pred) * len(ref)

        scores.append(distance / max(len(ref), len(pred), 1))

    return float(np.mean(scores))


def exact_match_rate(predictions, references):

    matches = 0

    for pred, ref in zip(predictions, references):

        if pred.strip() == ref.strip():
            matches += 1

    return matches / len(references)


def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    pred_str = processor.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    labels = np.where(
        labels != -100,
        labels,
        processor.tokenizer.pad_token_id
    )

    label_str = processor.batch_decode(
        labels,
        skip_special_tokens=True
    )

    cer_score = cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    wer_score = wer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    ned_score = normalized_edit_distance(
        pred_str,
        label_str
    )

    emr_score = exact_match_rate(
        pred_str,
        label_str
    )

    return {

        "CER": cer_score,
        "WER": wer_score,
        "NED": ned_score,
        "EMR": emr_score

    }

# Training Arguments

In [12]:
training_args = Seq2SeqTrainingArguments(

    output_dir="/content/checkpoints",

    overwrite_output_dir=True,

    num_train_epochs=20,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=5e-5,

    weight_decay=0.01,

    warmup_ratio=0.10,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    save_total_limit=2,

    load_best_model_at_end=True,

    metric_for_best_model="CER",

    greater_is_better=False,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    remove_unused_columns=False,

    report_to="none"

)

# Trainer

In [13]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    tokenizer=processor,

    data_collator=default_data_collator,

    compute_metrics=compute_metrics,

    callbacks=[

        EarlyStoppingCallback(
            early_stopping_patience=3
        )

    ]

)

# Train

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Cer,Wer,Ned,Emr
0,No log,5.033697,1.547597,1.000000,0.933455,0.000000
2,No log,3.368008,1.575786,1.222222,0.847516,0.000000
4,No log,3.204283,1.273105,1.510288,0.825195,0.000000
6,5.121000,2.779464,0.920055,1.061728,0.809057,0.000000
8,5.121000,2.578313,0.990296,1.207819,0.783463,0.000000
10,2.786300,2.566187,0.816081,1.080247,0.773174,0.000000


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

TrainOutput(global_step=203, training_loss=3.938196999686105, metrics={'train_runtime': 3025.0149, 'train_samples_per_second': 0.959, 'train_steps_per_second': 0.119, 'total_flos': 1.193514879638569e+18, 'train_loss': 3.938196999686105, 'epoch': 10.972972972972974})

# Test Evaluation

In [15]:
results = trainer.evaluate(test_dataset)

print(results)

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


{'eval_loss': 2.9064667224884033, 'eval_CER': 0.8569364161849711, 'eval_WER': 1.0328947368421053, 'eval_NED': 0.8369358477882124, 'eval_EMR': 0.0, 'eval_runtime': 85.8984, 'eval_samples_per_second': 0.373, 'eval_steps_per_second': 0.093, 'epoch': 10.972972972972974}


# Save Best Model

In [16]:

SAVE_DIR = "/content/Urdu_TrOCR"

trainer.save_model(SAVE_DIR)

processor.save_pretrained(SAVE_DIR)

print("Model Saved Successfully.")

Model Saved Successfully.


# Gradio Interface for Urdu OCR

In [18]:
import gradio as gr

with gr.Blocks(title="Urdu OCR") as demo:

    gr.Markdown("# 📝 Urdu OCR using TrOCR")
    gr.Markdown("Upload an Urdu text image to extract text.")

    with gr.Row():
        image_input = gr.Image(type="pil", label="Input Image")

    output = gr.Textbox(
        label="Extracted Text",
        lines=6
    )

    btn = gr.Button("Extract Text")

    btn.click(
        fn=predict_text,
        inputs=image_input,
        outputs=output
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20a19427568a78c8f2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
